# Garbage Classification — Full Experiment Pipeline on Google Colab

**Full workflow** theo `experiments/EXPERIMENTS_GUIDE.md`, chạy trên GPU miễn phí.

**Bước 0**: Đặt runtime thành GPU
- Nhấp **Runtime** → **Change runtime type** → GPU → **Save**

## 1. Setup Environment

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

### 1.2 Clone Repository

In [ ]:
# Nếu repo public
!git clone https://github.com/dinhhoang235/Garbage_Classification_Project.git
%cd Garbage_Classification_Project
print("✓ Repo cloned")

# Nếu private: upload zip vào Drive, sau đó uncomment và chạy
# !cp /content/drive/MyDrive/Garbage_Classification_Project.zip .
# !unzip -q Garbage_Classification_Project.zip
# %cd Garbage_Classification_Project

### 1.3 Check GPU & Install Dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -r requirements.txt
print("✓ Dependencies installed")

### 1.4 Download Data từ Google Drive

In [ ]:
# Trước tiên, upload data vào Drive (ví dụ: MyDrive/garbage_data/)
# File structure: MyDrive/garbage_data/[class_name]/img.jpg

!mkdir -p data/raw/original
!cp -r /content/drive/MyDrive/garbage_data/* data/raw/original/ 2>/dev/null || echo "⚠ Data chưa upload hoặc path khác"
!ls data/raw/original/ | head -20

## 2. Prepare Data (Preprocess & Split)

In [ ]:
# Tạo train/val/test split (70/15/15) từ dữ liệu gốc
!python3 src/preprocess.py
print("✓ Data preprocessed")

## 3. Run Batch Experiments

Chạy batch training: 3 configs × 30 epochs mỗi

### Batch run: 3 models × 30 epochs

In [ ]:
print("Chạy FULL EXPERIMENTS: 3 configs × 30 epochs")
print("="*60)
print("Models: MobileNetV1, MobileNetV3Large, EfficientNetV2S")
print("Thời gian dự tính: ~2-3 giờ (tùy GPU)\n")

In [ ]:
# Batch run: tất cả 3 models
!python3 tools/run_experiments.py --configs \
  experiments/configs/mobilenetv1_baseline.yaml \
  experiments/configs/mobilenetv3_large.yaml \
  experiments/configs/efficientnetv2s.yaml

In [ ]:
# Copy toàn bộ results về Drive
!mkdir -p /content/drive/MyDrive/garbage_runs/full_experiments
!cp -r model/weights /content/drive/MyDrive/garbage_runs/full_experiments/
!cp -r experiments/results_summary.csv /content/drive/MyDrive/garbage_runs/full_experiments/ 2>/dev/null || echo "Results file updated"
!cp -r experiments/plots /content/drive/MyDrive/garbage_runs/full_experiments/ 2>/dev/null || echo "Plots folder pending"
print("✓ Results & checkpoints lưu vào Drive")

## 4. Evaluate Models on Test Set

Sinh classification report và confusion matrix cho từng model

In [ ]:
# Đánh giá từng model lưu trong model/weights/
import os
import glob

models = glob.glob('model/weights/*_best.keras')
print(f"Tìm thấy {len(models)} model(s)")
for m in models:
    print(f"  - {m}")

In [ ]:
# Evaluate từng model
for model_path in models:
    print(f"\nEvaluating {model_path}...")
    !python3 src/evaluate.py \
      --model_path {model_path} \
      --base_dir data/raw/original \
      --img_size 224 224 \
      --batch_size 32

## 5. Generate Comparison Plots

In [ ]:
# Vẽ biểu đồ so sánh (nếu có results_summary.csv)
import os

if os.path.exists('experiments/results_summary.csv'):
    !python3 tools/plot_results.py --results_path experiments/results_summary.csv
    print("✓ Plots generated → experiments/plots/")
else:
    print("⚠ results_summary.csv chưa có. Chạy batch run (Cell 3B) trước.")

## 6. View Training History (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir . --port 6006

## 7. Save Everything to Drive

In [ ]:
# Backup final results & plots
!mkdir -p /content/drive/MyDrive/garbage_runs/final_results

# Copy results
!cp -r experiments/results_summary.csv /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null
!cp -r experiments/plots /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null
!cp -r reports /content/drive/MyDrive/garbage_runs/final_results/ 2>/dev/null

print("✓ All results saved to Drive: MyDrive/garbage_runs/final_results")

## 8. Quick Analysis

In [ ]:
# Đọc và hiển thị results summary
import pandas as pd

try:
    df = pd.read_csv('experiments/results_summary.csv')
    print("📊 Results Summary:")
    print(df)
    
    # Tìm best model
    best_idx = df['test_accuracy'].idxmax()
    print(f"\n🏆 Best model: {df.loc[best_idx, 'backbone']} (Accuracy: {df.loc[best_idx, 'test_accuracy']:.4f})")
except:
    print("⚠ Chưa có results_summary.csv")

## Tips

- ✓ Chạy batch experiments: 3 models × 30 epochs (~2-3 giờ)
- ✓ Luôn backup kết quả vào Drive trước khi session timeout
- ✓ Kiểm tra GPU memory với `nvidia-smi` nếu OOM
- ✓ Dùng Colab Pro nếu cần chạy lâu (Colab miễn phí giới hạn ~12h)